## Comparative Analysis of Regression Models on the California Housing Dataset

This project focuses on data preprocessing, implementing four machine learning models, and providing a final comparative analysis of their performance on the California Housing dataset.


## Import Libraries

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


## Load Dataset

## Load the Dataset

In this step, we load the California Housing dataset and preview the first few rows.


## Read CSV

In [22]:
df = pd.read_csv('housing.csv')
df.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


# Data Preprocessing

## 1. Data Preprocessing

In this section, we handle missing values, remove outliers, and convert categorical variables into numerical format for model training.


# Fill Missing Values

In [23]:
# Fill missing values in the total_bedrooms column using the median
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())


# Remove Outliers

In [24]:
# Remove outliers using the IQR method
num_cols = ['total_rooms', 'total_bedrooms', 'population', 'households', 'median_house_value']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]


## Encode Categorical Variables

In [25]:
# Convert categorical variables into dummy/indicator variables
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

# Display dataset information after preprocessing
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 17190 entries, 0 to 20639
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   17190 non-null  float64
 1   latitude                    17190 non-null  float64
 2   housing_median_age          17190 non-null  float64
 3   total_rooms                 17190 non-null  float64
 4   total_bedrooms              17190 non-null  float64
 5   population                  17190 non-null  float64
 6   households                  17190 non-null  float64
 7   median_income               17190 non-null  float64
 8   median_house_value          17190 non-null  float64
 9   ocean_proximity_INLAND      17190 non-null  bool   
 10  ocean_proximity_ISLAND      17190 non-null  bool   
 11  ocean_proximity_NEAR BAY    17190 non-null  bool   
 12  ocean_proximity_NEAR OCEAN  17190 non-null  bool   
dtypes: bool(4), float64(9)
memory usage:

## Modeling

## 2. Modeling

In this part, we split the dataset into training and testing sets, standardize the features, and train four different models:
- Simple Linear Regression
- Multiple Linear Regression
- Polynomial Regression
- Logistic Regression


## Split Data and Scale Features

In [26]:
# Define features and target variable
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Standardize the feature set
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


## Simple Linear Regression

In [27]:
# A. Simple Linear Regression using median_income as the predictor
X_train_s = X_train[:, [7]]   # median_income
X_test_s = X_test[:, [7]]

lr_s = LinearRegression().fit(X_train_s, y_train)


## Multiple Linear Regression

In [28]:
# B. Multiple Linear Regression using all features
lr_m = LinearRegression().fit(X_train, y_train)


## Polynomial Regression

In [29]:
# C. Polynomial Regression with degree 2
poly = PolynomialFeatures(degree=2)

X_train_p = poly.fit_transform(X_train)
X_test_p = poly.transform(X_test)

lr_p = LinearRegression().fit(X_train_p, y_train)


## Logistic Regression

In [30]:
# D. Logistic Regression for binary classification
# Houses with value >= 200000 are labeled as 1, otherwise 0
y_bin = (y >= 200000).astype(int)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_scaled, y_bin, test_size=0.2, random_state=42
)

log_r = LogisticRegression().fit(X_tr_c, y_tr_c)


## Evaluation and Comparison

## 3. Evaluation and Comparison

In this section, we evaluate the regression models using:
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- R-squared (R2)

For Logistic Regression, we evaluate:
- Accuracy
- F1-Score


## Regression Evaluation Function

In [31]:
def get_reg_metrics(y_true, y_pred):
    return [
        mean_squared_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        mean_absolute_error(y_true, y_pred),
        r2_score(y_true, y_pred)
    ]


## Compare Regression Models

In [32]:
res_s = get_reg_metrics(y_test, lr_s.predict(X_test_s))
res_m = get_reg_metrics(y_test, lr_m.predict(X_test))
res_p = get_reg_metrics(y_test, lr_p.predict(X_test_p))

table = pd.DataFrame(
    [res_s, res_m, res_p],
    columns=['MSE', 'RMSE', 'MAE', 'R2'],
    index=['Simple Linear', 'Multiple Linear', 'Polynomial']
)

print("Regression Models Comparison:")
display(table)


Regression Models Comparison:


,MSE,RMSE,MAE,R2
Simple Linear,5.502083e+09,74176.028442,56157.794541,0.416596
Multiple Linear,3.513876e+09,59277.952283,43393.118791,0.627412
Polynomial,2.853244e+09,53415.765030,38192.783248,0.697461


## Classification Metrics

In [33]:
# Evaluate Logistic Regression
y_pred_c = log_r.predict(X_te_c)

print(f"\nLogistic Regression Accuracy: {accuracy_score(y_te_c, y_pred_c):.4f}")
print(f"F1-Score: {f1_score(y_te_c, y_pred_c):.4f}")



Logistic Regression Accuracy: 0.8447
F1-Score: 0.7979


## Final Analysis

## 4. Final Analysis

Polynomial Regression usually achieves the best R2 score because it can capture nonlinear relationships in the data more effectively than linear models.

Logistic Regression also shows reasonable performance for classifying high-value houses, making it a useful approach when the problem is transformed into a binary classification task.
